# 🧠🎯 Warm-start + Benchmark com PokerBench

**Celula unica - Run all (GPU).** A rede primeiro **clona as decisoes de um solver** (joga GTO) e depois faz **self-play** pra refinar. Medimos a **concordancia com o solver** (benchmark GTO) antes/depois.

Reaproveita o **`encode()`** do self-play (via o parser testado `poker_arena.ml.pokerbench`) - entao os pesos do warm-start transferem pro treino.

**Pre-requisitos (Secrets do Colab):** `GH_TOKEN` (repo privado) e `HF_TOKEN`.

In [ ]:
# ============================================================
# WARM-START + BENCHMARK com PokerBench (6-max NLHE, decisoes de solver/GTO)
# Pre-req (Secrets do Colab): GH_TOKEN (repo privado) e HF_TOKEN (dataset + push)
# Celula unica: Run all. Roda em GPU.
# ============================================================
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
gh = userdata.get('GH_TOKEN')
assert gh and os.environ.get('HF_TOKEN'), 'Configure GH_TOKEN e HF_TOKEN nos Secrets do Colab.'
!rm -rf poker-arena
!git clone -q https://{gh}@github.com/FELIPEACASTRO/poker-arena.git
!pip install -q poker-arena/backend  # nao-editable: importavel sem reiniciar o kernel
!pip install -q stable-baselines3 sb3-contrib gymnasium onnx pandas huggingface_hub
print('Instalado. (NAO compartilhe a saida - contem o token)')

import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import gymnasium as gym
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from huggingface_hub import HfApi, hf_hub_download
from poker_arena.ml import FEATURE_SIZE, N_ACTIONS, SelfPlayEnv
from poker_arena.ml.pokerbench import featurize


def box(titulo, linhas):
    w = max([len(titulo)] + [len(s) for s in linhas]) if linhas else len(titulo)
    print('\n  +' + '-'*(w+2) + '+'); print('  | ' + titulo.ljust(w) + ' |')
    print('  +' + '-'*(w+2) + '+')
    for s in linhas: print('  | ' + s.ljust(w) + ' |')
    print('  +' + '-'*(w+2) + '+')


# -------------------- CONFIG (aumente para treinar mais) --------------------
N_BC = 120_000          # registros do PokerBench no warm-start
BC_EPOCHS = 10
SELFPLAY_STEPS = 1_000_000
HF_MODEL = 'felipesp1983/poker-arena-expert'

box('WARM-START + BENCHMARK com PokerBench', [
    'A rede primeiro CLONA as decisoes de um solver (joga GTO), depois faz',
    'self-play pra refinar. Medimos a concordancia com o solver antes e depois.',
    'O BC e so o ponto de partida; o self-play depois POLE o agente.',
])

# -------------------- 1. carrega o PokerBench do HF --------------------
api = HfApi(token=os.environ['HF_TOKEN'])
csvs = [f for f in api.list_repo_files('RZ412/PokerBench', repo_type='dataset') if f.endswith('.csv')]
def grab(kw, split):
    fn = next(f for f in csvs if kw in f.lower() and split in f.lower())
    return pd.read_csv(hf_hub_download('RZ412/PokerBench', fn, repo_type='dataset',
                                       token=os.environ['HF_TOKEN']))
train = pd.concat([grab('preflop','train'), grab('postflop','train')], ignore_index=True)
test  = pd.concat([grab('preflop','test'),  grab('postflop','test')],  ignore_index=True)
train = train.sample(min(N_BC, len(train)), random_state=0)
box('POKERBENCH CARREGADO', [f'treino (warm-start): {len(train):,} registros',
                             f'teste (benchmark)  : {len(test):,} registros'])

# -------------------- 2. parse -> features (REUSA o nosso encode) --------------------
def to_arrays(df):
    Xs, Ms, Ys = [], [], []
    for r in df.to_dict('records'):
        try:
            f, m, y = featurize(r)
        except Exception:
            continue
        Xs.append(f); Ms.append(m); Ys.append(y)
    return np.asarray(Xs, np.float32), np.asarray(Ms, bool), np.asarray(Ys, np.int64)
Xtr, Mtr, Ytr = to_arrays(train)
Xte, Mte, Yte = to_arrays(test)
box('FEATURES PRONTAS', [f'treino: {Xtr.shape}  teste: {Xte.shape}',
                         'mesma codificacao do self-play (encode reaproveitado -> pesos transferem)'])

# -------------------- 3. monta o PPO --------------------
class GymPokerEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.env = SelfPlayEnv(n_players=6)
        self.observation_space = spaces.Box(0.0, 5.0, (FEATURE_SIZE,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self._m = [True]*N_ACTIONS
    def reset(self, *, seed=None, options=None):
        o, self._m = self.env.reset(); return np.asarray(o, np.float32), {}
    def step(self, a):
        o, self._m, r, d = self.env.step(int(a)); return np.asarray(o, np.float32), float(r), bool(d), False, {}
    def action_masks(self):
        return np.asarray(self._m, bool)

env = GymPokerEnv()
model = MaskablePPO('MlpPolicy', env, n_steps=2048, batch_size=256, ent_coef=0.01,
                    gamma=0.99, gae_lambda=0.95, policy_kwargs=dict(net_arch=[256,256]),
                    verbose=0, device='auto')

def agreement(X, M, Y):
    if len(X) == 0: return 0.0
    a, _ = model.predict(X, action_masks=M, deterministic=True)
    return float((np.asarray(a) == Y).mean()) * 100.0

box('BENCHMARK ANTES (politica aleatoria, sem treino)',
    [f'concordancia com o solver: {agreement(Xte, Mte, Yte):.1f}%'])

# -------------------- 4. WARM-START: behavioral cloning na politica do PPO --------------------
device = model.device  # GPU no Colab -> os tensores do BC precisam ir junto
Xt = torch.tensor(Xtr, device=device)
Yt = torch.tensor(Ytr, device=device)
actor = list(model.policy.mlp_extractor.policy_net.parameters()) + list(model.policy.action_net.parameters())
opt = torch.optim.Adam(actor, lr=1e-3)
box('WARM-START - clonando as decisoes do solver...',
    ['a rede aprende a jogar GTO ANTES de jogar contra si mesma'])
B = 1024
for ep in range(BC_EPOCHS):
    perm = torch.randperm(len(Xt), device=device); tot = 0.0
    for i in range(0, len(Xt), B):
        idx = perm[i:i+B]
        feats = model.policy.extract_features(Xt[idx])
        logits = model.policy.action_net(model.policy.mlp_extractor.forward_actor(feats))
        loss = F.cross_entropy(logits, Yt[idx])
        opt.zero_grad(); loss.backward(); opt.step(); tot += float(loss)*len(idx)
    print(f'   epoca {ep+1:>2}/{BC_EPOCHS}  perda={tot/len(Xt):.3f}  concordancia(teste)={agreement(Xte,Mte,Yte):.1f}%')

box('BENCHMARK APOS WARM-START',
    [f'concordancia com o solver: {agreement(Xte, Mte, Yte):.1f}%', '(deve subir MUITO vs aleatorio)'])

# -------------------- 5. self-play continua do warm-start --------------------
def opp(feats, mask):
    a, _ = model.predict(np.asarray(feats, np.float32), action_masks=np.asarray(mask, bool), deterministic=False)
    return int(a)
env.env.set_opponent(opp)
box('SELF-PLAY a partir do warm-start...',
    [f'{SELFPLAY_STEPS:,} passos; a IA refina o que aprendeu do solver'])
model.learn(total_timesteps=SELFPLAY_STEPS)
model.save('poker_ppo_warmstart')
box('BENCHMARK FINAL (apos self-play)',
    [f'concordancia com o solver: {agreement(Xte, Mte, Yte):.1f}%'])

# -------------------- 6. export ONNX + publica no HF --------------------
class OnnxPolicy(torch.nn.Module):
    def __init__(self, p):
        super().__init__()
        self.fe, self.mlp, self.an = p.features_extractor, p.mlp_extractor, p.action_net
    def forward(self, obs):
        f = self.fe(obs); lp, _ = self.mlp(f); return self.an(lp)
torch.onnx.export(OnnxPolicy(model.policy.to('cpu').eval()), torch.zeros(1, FEATURE_SIZE),
                  'poker_expert.onnx', input_names=['obs'], output_names=['logits'],
                  dynamic_axes={'obs': {0: 'batch'}, 'logits': {0: 'batch'}},
                  opset_version=17, dynamo=False)
api.create_repo(HF_MODEL, repo_type='model', exist_ok=True, private=True)
api.upload_file(path_or_fileobj='poker_expert.onnx', path_in_repo='poker_expert.onnx',
                repo_id=HF_MODEL, repo_type='model')
box('PUBLICADO NO HF HUB', [f'modelo: {HF_MODEL}',
                            'A Fase 4 (MLBot no backend) baixa este poker_expert.onnx daqui.'])
